# Variable Construction and Preliminary Models

**<u>See variable info, notes on IV, and update summary at bottom before proceeding</u>**

This notebook prepares the data and constructs the main variables used in the preliminary analysis of food insecurity among seniors in Clark County.

The analysis proceeds in the following stages:

1. Load and prepare the original survey data.
2. Create parallel full and deduplicated analysis samples.
3. Verify key identifier variables and duplicate structure.
4. Construct food insecurity and very low food security outcome variables.
5. Construct and evaluate financial literacy variables.
6. Examine missingness, distributions, and relationships among the financial literacy measures.
7. Create alternative financial literacy indices for later comparison.
8. Inspect and prepare basic control variables for adjusted models.
9. Estimate preliminary baseline and controlled models of food insecurity and very low food security.

Variable definitions are applied consistently across both analysis samples so results can be compared between the full and deduplicated datasets. The overall goal is to evaluate whether the main empirical relationships remain stable across different sample definitions and financial literacy measurement approaches.

In [1]:
# Import necessary libraries for data analysis and preprocessing
import pandas as pd
import numpy as np

# Display settings for wider survey datasets
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# Confirm package versions for reproducibility
print("pandas version:", pd.__version__)
print("numpy version:", np.__version__)

pandas version: 3.0.3
numpy version: 2.4.5


## Load Data and Prepare Analysis Samples

We begin by loading the original survey dataset. Because the professor recommended comparing results across the original and deduplicated data, this notebook prepares two parallel samples:

1. `df_full`: the original dataset.
2. `df_unique`: the deduplicated comparison dataset.

Variable construction rules will be applied consistently to both samples.

In [2]:
# Load original survey dataset
df_full = pd.read_csv("../data/Clark_County_Senior_FIS_Data_N_800.csv")

# Create deduplicated comparison sample
df_unique = df_full.drop_duplicates()

# Compare sample dimensions
print("Full sample shape:", df_full.shape)
print("Deduplicated sample shape:", df_unique.shape)
print("Rows removed:", len(df_full) - len(df_unique))

Full sample shape: (800, 112)
Deduplicated sample shape: (555, 112)
Rows removed: 245


The original dataset contains 800 rows and 112 columns. After removing exact duplicate rows across all columns, the deduplicated comparison sample contains 555 rows, meaning 245 duplicate rows were removed.

Both samples are retained for analysis. The full sample preserves the original data structure, while the deduplicated sample allows us to test whether the empirical results are sensitive to repeated observations.

## Verify Key Identifier Columns

Before proceeding with variable construction, we verify that the key identifier columns are present in both analysis samples. Key identifier columns are variables that help distinguish one survey record from another. In this dataset, `_survnum` and `pin` are especially important because they help confirm whether repeated rows are true duplicate records rather than merely similar responses.

In [3]:
# Verify identifier columns used in duplicate analysis
identifier_columns = ["_survnum", "pin"]

for col in identifier_columns:
    print(f"{col} in full sample:", col in df_full.columns)
    print(f"{col} in deduplicated sample:", col in df_unique.columns)

_survnum in full sample: True
_survnum in deduplicated sample: True
pin in full sample: True
pin in deduplicated sample: True


Both `_survnum` and `pin` are present in the original and deduplicated datasets. Because the duplicate analysis was conducted across all 112 columns, including these identifier variables, the duplicate removal process reflects exact repeated observations rather than records that simply share some similar responses. In this context, a “true duplicate” means that every recorded value in the row is identical across two or more observations, meaning the duplicated rows are exact copies of one another across the full survey record.

## Construct Food Insecurity Outcome Variables

The food security outcomes are constructed from the six USDA-style adult food security items in the survey: `q17` through `q22`.

Each item is first recoded into an affirmative indicator, where 1 indicates evidence of food insecurity and 0 indicates no affirmative evidence. The six indicators are then summed into a raw food security score ranging from 0 to 6.

Following the standard six-item classification logic:

- Raw scores of 0–1 indicate food secure or marginal food security.
- Raw scores of 2–4 indicate low food security.
- Raw scores of 5–6 indicate very low food security.

This produces two binary outcome variables:

- `food_insecure`: equals 1 when raw score is 2 or higher.
- `very_low_security`: equals 1 when raw score is 5 or higher.

In [4]:
# Construct food security affirmative indicators
def construct_food_security_outcomes(df):
    df = df.copy()
    
    df["fs_q17_affirm"] = df["q17"].isin([1, 2]).astype(int)
    df["fs_q18_affirm"] = df["q18"].isin([1, 2]).astype(int)
    df["fs_q19_affirm"] = (df["q19"] == 1).astype(int)
    df["fs_q20_affirm"] = df["q20"].isin([1, 2]).astype(int)
    df["fs_q21_affirm"] = (df["q21"] == 1).astype(int)
    df["fs_q22_affirm"] = (df["q22"] == 1).astype(int)
    
    fs_items = [
        "fs_q17_affirm",
        "fs_q18_affirm",
        "fs_q19_affirm",
        "fs_q20_affirm",
        "fs_q21_affirm",
        "fs_q22_affirm"
    ]
    
    df["food_security_raw_score"] = df[fs_items].sum(axis=1)
    df["food_insecure"] = (df["food_security_raw_score"] >= 2).astype(int)
    df["very_low_security"] = (df["food_security_raw_score"] >= 5).astype(int)
    
    return df

df_full = construct_food_security_outcomes(df_full)
df_unique = construct_food_security_outcomes(df_unique)

In [5]:
# Compare food security outcome distributions across analysis samples
for name, data in [("Full sample", df_full), ("Deduplicated sample", df_unique)]:
    
    print(f"\n{name}")
    print("-" * 50)
    
    print("\nFood insecurity rate:")
    print((data["food_insecure"]
           .value_counts(normalize=True)
           .sort_index()
           .round(3)))
    
    print("\nVery low food security rate:")
    print((data["very_low_security"]
           .value_counts(normalize=True)
           .sort_index()
           .round(3)))


Full sample
--------------------------------------------------

Food insecurity rate:
food_insecure
0    0.701
1    0.299
Name: proportion, dtype: float64

Very low food security rate:
very_low_security
0    0.901
1    0.099
Name: proportion, dtype: float64

Deduplicated sample
--------------------------------------------------

Food insecurity rate:
food_insecure
0    0.708
1    0.292
Name: proportion, dtype: float64

Very low food security rate:
very_low_security
0    0.901
1    0.099
Name: proportion, dtype: float64


### Verify Nested Food Security Outcomes

The following crosstabs confirm that all respondents classified as `very_low_security = 1` are also classified as `food_insecure = 1` in both analysis samples.

In [6]:
# Verify that all respondents classified as very low food security
# are also classified as food insecure in both analysis samples

from IPython.display import display

print("Deduplicated sample")
print("-" * 50)

display(
    pd.crosstab(
        df_unique["food_insecure"],
        df_unique["very_low_security"]
    )
)

print("\nFull sample")
print("-" * 50)

display(
    pd.crosstab(
        df_full["food_insecure"],
        df_full["very_low_security"]
    )
)

Deduplicated sample
--------------------------------------------------


very_low_security,0,1
food_insecure,,
0,393,0
1,107,55



Full sample
--------------------------------------------------


very_low_security,0,1
food_insecure,,
0,561,0
1,160,79


The crosstabs confirm that the outcome variables are logically nested across both analysis samples. In both the full and deduplicated datasets, all respondents classified as `very_low_security = 1` are also classified as `food_insecure = 1`, with zero contradictory cases observed.

The prevalence rates are also highly similar across the two samples. In the full sample, 29.9% of respondents are classified as food insecure and 9.9% as very low food security. In the deduplicated sample, the corresponding rates are 29.2% and 9.9%.

The near-identical distributions across both samples suggest that duplicate observations do not materially alter the overall structure of the primary outcome variables. This supports retaining both datasets for later robustness comparison in the regression analysis.

**<u>Definition:</u>** `food_insecure` represents the broader food insecurity category, while `very_low_security` represents the more severe subgroup within it. Therefore, every respondent classified as very low food security is also food insecure, but not every food insecure respondent is classified as very low food security.

## Construct Financial Literacy Variables

The survey includes several related but distinct measures of financial functioning. These include retrospective confidence in parents’ or guardians’ financial capability, current self-rated confidence in managing day-to-day financial matters, and current financial behaviors or practices. Because these measures capture related but not identical concepts, the index construction below treats them carefully rather than assuming they are interchangeable.

The relevant variables come from three sections of the survey:

- `q43`: retrospective confidence in parents'/guardians' financial capability during childhood.
- `q44`: current self-rated confidence in managing day-to-day financial matters.
- `q45a` through `q45h`: financial behavior and financial practice items.

Before constructing composite measures or index candidates, we first inspect the distributions and coding direction of the underlying variables.

In [7]:
# Financial literacy and capability variables
finlit_vars = [
    "q43",
    "q44",
    "q45a",
    "q45b",
    "q45c",
    "q45d",
    "q45e",
    "q45f",
    "q45g",
    "q45h"
]

# Compare distributions across analysis samples
for name, data in [("Full sample", df_full), ("Deduplicated sample", df_unique)]:
    
    print(f"\n{name}")
    print("-" * 50)
    
    for var in finlit_vars:
        print(f"\n{var}")
        print(data[var].value_counts(dropna=False).sort_index())


Full sample
--------------------------------------------------

q43
q43
0    140
1     55
2     48
3     73
4    142
5    342
Name: count, dtype: int64

q44
q44
1     30
2     35
3     76
4    170
5    489
Name: count, dtype: int64

q45a
q45a
1    123
2    138
3     33
4    102
5    177
6    227
Name: count, dtype: int64

q45b
q45b
1     89
2    164
3     33
4     99
5    155
6    260
Name: count, dtype: int64

q45c
q45c
1     30
2     97
3     27
4     92
5    174
6    380
Name: count, dtype: int64

q45d
q45d
1     13
2     54
3     16
4     93
5    122
6    502
Name: count, dtype: int64

q45e
q45e
1     14
2     32
3      8
4     38
5    102
6    606
Name: count, dtype: int64

q45f
q45f
1     29
2     41
3     13
4     82
5    107
6    528
Name: count, dtype: int64

q45g
q45g
1     47
2    115
3     23
4    163
5    140
6    312
Name: count, dtype: int64

q45h
q45h
1     71
2     81
3     24
4     69
5    104
6    451
Name: count, dtype: int64

Deduplicated sample
------------------

The financial literacy and financial capability variables appear internally consistent with the survey codebook and contain no unexpected response values.

Several important distributional patterns emerge:

- `q43` differs conceptually from the other variables because it measures retrospective perceptions of parents’ or guardians’ financial capability during childhood rather than the respondent’s current financial behavior or confidence. In addition, the variable contains a substantial number of `0 = Don't Know` responses. Because of both the conceptual distinction and response structure, this variable may require separate handling or exclusion from composite index construction.
- `q44` is heavily concentrated in the upper confidence categories, indicating that most respondents report relatively high confidence in managing day-to-day financial matters.
- The `q45` financial behavior variables are strongly concentrated at the highest response category (`6 = I am doing this all of the time`), suggesting limited variation across several behavioral measures.

For the current confidence and financial behavior variables, higher values generally correspond to stronger financial confidence or more frequent positive financial practices. This supports using these variables in standardized index construction, while continuing to treat `q43` separately because of the concerns noted above.

These patterns are important because highly concentrated variables may contribute less distinguishing information in later index construction and regression analysis.

### Missingness in Financial Literacy Variables

We now examine missingness across the financial literacy variables before constructing composite measures and indices. This step is important because missing responses can reduce the effective sample size in later analysis, particularly during PCA estimation and regression modeling. Missingness patterns may also indicate data quality concerns or inconsistencies across survey items.

In [8]:
# Check missingness in financial literacy variables across both samples
for name, data in [("Full sample", df_full), ("Deduplicated sample", df_unique)]:
    missing_summary = pd.DataFrame({
        "missing_count": data[finlit_vars].isna().sum(),
        "missing_percent": data[finlit_vars].isna().mean() * 100
    })

    print(f"\n{name}")
    print("-" * 50)
    print(missing_summary)


Full sample
--------------------------------------------------
      missing_count  missing_percent
q43               0              0.0
q44               0              0.0
q45a              0              0.0
q45b              0              0.0
q45c              0              0.0
q45d              0              0.0
q45e              0              0.0
q45f              0              0.0
q45g              0              0.0
q45h              0              0.0

Deduplicated sample
--------------------------------------------------
      missing_count  missing_percent
q43               0              0.0
q44               0              0.0
q45a              0              0.0
q45b              0              0.0
q45c              0              0.0
q45d              0              0.0
q45e              0              0.0
q45f              0              0.0
q45g              0              0.0
q45h              0              0.0


The financial literacy variables contain no missing observations in either the full or deduplicated samples. This is beneficial for later index construction because both the standardized Z-score index and the PCA-based alternative index can be estimated without losing observations due to incomplete financial literacy responses.

Because the variables are measured on related but not identical scales and exhibit different response distributions, the later index construction standardizes the variables before aggregation, and PCA estimation.

## Construct Preliminary Financial Literacy Measures

Before constructing the standardized and PCA-based financial literacy indices, we first create simple descriptive measures from the financial behavior items. These preliminary measures are useful for understanding the scale, distribution, and concentration of responses before moving to later index construction.

At this stage, these measures should be interpreted as descriptive benchmarks rather than final index variables. Because the underlying items exhibit different response distributions and levels of variation, later sections construct standardized financial literacy indices for use in modeling.

The preliminary measures include:

1. `q44`: current self-rated confidence in managing day-to-day financial matters.
2. `q45_mean`: average response across the eight financial behavior items.
3. `q45_sum`: total response score across the eight financial behavior items.

Higher values indicate greater reported financial confidence or more frequent use of positive financial practices.

In [9]:
# Construct preliminary financial literacy measures
q45_items = ["q45a", "q45b", "q45c", "q45d", "q45e", "q45f", "q45g", "q45h"]

def construct_preliminary_finlit_measures(df):
    df = df.copy()
    
    df["q45_mean"] = df[q45_items].mean(axis=1)
    df["q45_sum"] = df[q45_items].sum(axis=1)
    
    return df

df_full = construct_preliminary_finlit_measures(df_full)
df_unique = construct_preliminary_finlit_measures(df_unique)

In [10]:
# Summarize preliminary financial literacy measures
for name, data in [("Full sample", df_full), ("Deduplicated sample", df_unique)]:
    
    print(f"\n{name}")
    print("-" * 50)
    
    print(data[["q44", "q45_mean", "q45_sum"]].describe().round(2))


Full sample
--------------------------------------------------
          q44  q45_mean  q45_sum
count  800.00    800.00   800.00
mean     4.32      4.74    37.93
std      1.06      1.05     8.42
min      1.00      1.12     9.00
25%      4.00      4.12    33.00
50%      5.00      5.00    40.00
75%      5.00      5.50    44.00
max      5.00      6.00    48.00

Deduplicated sample
--------------------------------------------------
          q44  q45_mean  q45_sum
count  555.00    555.00   555.00
mean     4.33      4.76    38.06
std      1.03      1.05     8.37
min      1.00      1.12     9.00
25%      4.00      4.12    33.00
50%      5.00      5.00    40.00
75%      5.00      5.50    44.00
max      5.00      6.00    48.00


These preliminary mean and sum measures are useful descriptive benchmarks, but they are not used as the primary financial literacy indices in later modeling. Because the underlying items exhibit different response distributions and levels of dispersion, later index construction standardizes the variables before aggregation and PCA estimation.

The standardized Z-score index and the PCA-based alternative index therefore provide more comparable summary measures of financial literacy while preserving the underlying correlation structure among the financial confidence and financial behavior variables.

## Examine Relationships Among Financial Literacy Variables

Before constructing the preferred Z-score index and PCA-based alternative index, we examine how the financial literacy variables relate to one another. This helps assess whether the items share enough common structure to support index construction, while also showing whether they capture distinct dimensions of financial confidence and financial behavior.

This step is especially important for PCA because PCA relies on the correlation structure among variables.

In [11]:
# Examine correlations among financial literacy variables
finlit_corr_vars = ["q44"] + q45_items

for name, data in [("Full sample", df_full), ("Deduplicated sample", df_unique)]:
    
    print(f"\n{name}")
    print("-" * 50)
    
    corr_matrix = data[finlit_corr_vars].corr()
    display(corr_matrix.round(2))


Full sample
--------------------------------------------------


,q44,q45a,q45b,q45c,q45d,q45e,q45f,q45g,q45h
q44,1.00,0.37,0.31,0.17,0.27,0.36,0.22,0.00,0.17
q45a,0.37,1.00,0.61,0.38,0.34,0.34,0.23,0.23,0.37
q45b,0.31,0.61,1.00,0.61,0.43,0.36,0.24,0.38,0.44
q45c,0.17,0.38,0.61,1.00,0.54,0.40,0.28,0.40,0.40
q45d,0.27,0.34,0.43,0.54,1.00,0.51,0.29,0.33,0.41
q45e,0.36,0.34,0.36,0.40,0.51,1.00,0.50,0.23,0.35
q45f,0.22,0.23,0.24,0.28,0.29,0.50,1.00,0.27,0.28
q45g,0.00,0.23,0.38,0.40,0.33,0.23,0.27,1.00,0.52
q45h,0.17,0.37,0.44,0.40,0.41,0.35,0.28,0.52,1.00



Deduplicated sample
--------------------------------------------------


,q44,q45a,q45b,q45c,q45d,q45e,q45f,q45g,q45h
q44,1.00,0.36,0.27,0.17,0.29,0.35,0.22,0.04,0.22
q45a,0.36,1.00,0.59,0.39,0.34,0.35,0.24,0.20,0.38
q45b,0.27,0.59,1.00,0.60,0.44,0.37,0.24,0.35,0.45
q45c,0.17,0.39,0.60,1.00,0.55,0.42,0.29,0.39,0.41
q45d,0.29,0.34,0.44,0.55,1.00,0.50,0.28,0.35,0.42
q45e,0.35,0.35,0.37,0.42,0.50,1.00,0.50,0.24,0.38
q45f,0.22,0.24,0.24,0.29,0.28,0.50,1.00,0.28,0.30
q45g,0.04,0.20,0.35,0.39,0.35,0.24,0.28,1.00,0.46
q45h,0.22,0.38,0.45,0.41,0.42,0.38,0.30,0.46,1.00


The financial literacy and financial capability variables exhibit broadly positive correlations across both analysis samples. Most pairwise correlations fall within a moderate range, suggesting that the variables share meaningful common structure while still capturing somewhat distinct dimensions of financial confidence and financial behavior.

Importantly, the correlations are not so high as to suggest that the variables are redundant or measuring exactly the same construct. This is useful for later index construction because both Z-score aggregation and PCA benefit from variables that are related but not perfectly collinear.

The `q45` behavioral variables are generally more strongly correlated with one another than with `q44`, which is conceptually reasonable because `q44` measures self-reported confidence while the `q45` items measure reported financial practices and behaviors.

The correlation structures are also highly similar across the full and deduplicated samples, further supporting the stability of the underlying relationships in the data.

One notable exception is the weak relationship between `q44` and `q45g`, suggesting that self-reported financial confidence may not align equally with every specific financial behavior item.

### Construct Z-Score Financial Literacy Index

Before estimating PCA-based measures, we construct a standardized financial literacy index using Z-score normalization. This approach standardizes each variable onto a common scale before aggregation, preventing variables with larger dispersion or different response distributions from disproportionately influencing the final index.

The standardized index is treated as the primary financial literacy measure in this analysis because it preserves interpretability while accounting for differences in scale and variation across the underlying survey items.

The primary standardized index includes:

- `q44`: current self-rated financial confidence
- `q45a-q45h`: current financial behavior items

`q43` is excluded from the primary index because it measures retrospective perceptions of parents’ or guardians’ financial capability during childhood rather than the respondent’s current financial literacy or financial behavior.

Higher index values indicate relatively stronger financial confidence and more frequent positive financial practices compared to the overall sample.

In [12]:
from sklearn.preprocessing import StandardScaler

# Variables included in the standardized financial literacy index
zscore_vars = [
    "q44",
    "q45a",
    "q45b",
    "q45c",
    "q45d",
    "q45e",
    "q45f",
    "q45g",
    "q45h"
]

def construct_zscore_finlit_index(df):
    df = df.copy()

    scaler = StandardScaler()

    # Standardize variables
    z_values = scaler.fit_transform(df[zscore_vars])

    z_df = pd.DataFrame(
        z_values,
        columns=[f"{col}_z" for col in zscore_vars],
        index=df.index
    )

    # Attach standardized variables
    df = pd.concat([df, z_df], axis=1)

    # Create standardized financial literacy index
    df["finlit_z_index"] = z_df.mean(axis=1)

    return df

# Apply to both samples
df_full = construct_zscore_finlit_index(df_full)
df_unique = construct_zscore_finlit_index(df_unique)

# Summarize standardized financial literacy index
for name, data in [("Full sample", df_full), ("Deduplicated sample", df_unique)]:

    print(f"\n{name}")
    print("-" * 50)

    print(
        data["finlit_z_index"]
        .describe()
        .round(3)
    )


Full sample
--------------------------------------------------
count    800.000
mean      -0.000
std        0.649
min       -2.140
25%       -0.342
50%        0.169
75%        0.493
max        0.767
Name: finlit_z_index, dtype: float64

Deduplicated sample
--------------------------------------------------
count    555.000
mean      -0.000
std        0.651
min       -2.156
25%       -0.336
50%        0.177
75%        0.486
max        0.763
Name: finlit_z_index, dtype: float64


The standardized financial literacy index exhibits nearly identical distributions across the full and deduplicated samples, further suggesting that duplicate observations do not materially alter the overall structure of the financial literacy measures.

Because the index is constructed from standardized variables, the resulting measure is centered approximately around zero by construction. Positive values indicate above-average financial confidence and financial behavior relative to the sample, while negative values indicate below-average levels.

The standard deviation of approximately 0.65 indicates moderate variation across respondents. Although each individual item is standardized to have a standard deviation near 1, averaging multiple related items reduces the dispersion of the final index.

The index range is also informative. The minimum value is around -2.15, while the maximum is around 0.76, suggesting that there are more respondents substantially below the sample average than far above it. This is consistent with the earlier finding that many financial behavior variables are concentrated in the upper response categories.

Overall, the Z-score index provides a standardized and interpretable measure of relative financial literacy while preserving meaningful differences across respondents.

## Construct Principal Component Analysis (PCA) Financial Literacy Measures

After examining the distributions and correlation structure of the financial literacy variables, we next construct a principal component analysis (PCA) measure of financial literacy as an alternative index specification.

Principal component analysis is a dimensionality reduction technique that combines correlated variables into a smaller number of summary components. In this project, PCA is used to summarize the shared variation across current financial confidence and financial behavior variables. We retain the first principal component as a single PCA-based financial literacy index because the first component captures the largest amount of common variation across the included items.

The PCA specification includes `q44` and `q45a` through `q45h`.

`q43` is excluded from the PCA construction for both conceptual and measurement reasons. While `q44` and `q45a-q45h` measure current financial confidence and present financial behaviors, `q43` measures retrospective perceptions of parental or guardian financial capability during childhood. Additionally, `q43` contains a separate `0 = Don't Know` response category that is not directly comparable to the ordinal response structure used in the remaining financial literacy variables.

Because PCA is most appropriate when variables reflect related underlying constructs, the specification focuses on current financial confidence and financial behavior measures. This PCA-based index is used as an alternative measure to compare against the preferred Z-score financial literacy index.

In [13]:
# Import tools for standardization and PCA
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

### Standardize Variables for PCA

Because PCA is sensitive to variable scale and dispersion, the financial literacy variables are standardized before PCA estimation. Standardization places each variable on a common scale with mean 0 and standard deviation 1, preventing variables with larger variance from mechanically exerting greater influence on the principal components.

This step is especially important here because some financial behavior variables exhibit more dispersion than others and several variables are concentrated in the upper response categories. Standardization ensures that PCA reflects the shared correlation structure among variables rather than differences in raw scale.

In [14]:
# Define variables included in the initial PCA specification
pca_vars = ["q44"] + q45_items

# Standardize PCA variables and estimate first principal component
def construct_pca_finlit(df):
    df = df.copy()
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df[pca_vars])
    
    pca = PCA(n_components=1)
    df["finlit_pca1"] = pca.fit_transform(X_scaled)
    
    return df, pca

df_full, pca_full = construct_pca_finlit(df_full)
df_unique, pca_unique = construct_pca_finlit(df_unique)

### Estimate Initial PCA Financial Literacy Measure

The first principal component explains approximately 43% of the standardized variation across the included financial confidence and financial behavior variables in both analysis samples. This indicates that a substantial share of the variation across the variables can be summarized by a single common dimension, supporting the use of a one-component PCA financial literacy index.

In [15]:
# Compare explained variance across PCA specifications
for name, pca in [("Full sample", pca_full), ("Deduplicated sample", pca_unique)]:
    
    print(f"\n{name}")
    print("-" * 50)
    
    print("Explained variance ratio:")
    print(round(pca.explained_variance_ratio_[0], 3))


Full sample
--------------------------------------------------
Explained variance ratio:
0.429

Deduplicated sample
--------------------------------------------------
Explained variance ratio:
0.431


In [16]:
# Examine PCA component loadings
for name, pca in [("Full sample", pca_full), ("Deduplicated sample", pca_unique)]:
    
    print(f"\n{name}")
    print("-" * 50)
    
    loadings = pd.DataFrame({
        "Variable": pca_vars,
        "Component Loading": pca.components_[0]
    })
    
    display(loadings.round(3))


Full sample
--------------------------------------------------


,Variable,Component Loading
0,q44,0.228
1,q45a,0.339
2,q45b,0.393
3,q45c,0.375
4,q45d,0.366
5,q45e,0.351
6,q45f,0.275
7,q45g,0.291
8,q45h,0.347



Deduplicated sample
--------------------------------------------------


,Variable,Component Loading
0,q44,0.234
1,q45a,0.336
2,q45b,0.384
3,q45c,0.379
4,q45d,0.368
5,q45e,0.356
6,q45f,0.276
7,q45g,0.281
8,q45h,0.351


The first principal component is extracted from the standardized financial confidence and financial behavior variables. This component represents the dominant shared variation across the included measures and serves as the PCA-based alternative financial literacy index.

The strongest component loadings are concentrated among the behavioral variables (`q45b-q45e`), while `q44` exhibits a smaller loading. This suggests that the PCA index is driven more heavily by reported financial practices and behaviors than by self-reported financial confidence alone.

All component loadings are positive, indicating that higher values across the included variables contribute positively to the financial literacy index. As a result, higher PCA scores can be interpreted as reflecting greater financial confidence and stronger financial management behaviors.


The behavioral variables (`q45a`–`q45h`) generally load more strongly onto the first component than `q44`, suggesting that the PCA index is driven more heavily by reported financial practices than by self-reported financial confidence alone.

The explained variance ratios and component loadings are nearly identical across the full and deduplicated samples, providing strong evidence that the PCA structure is stable and not materially influenced by duplicate observations.

### Examine PCA Eigenvalues

We next examine the eigenvalues associated with the first principal component. In PCA, an eigenvalue measures how much standardized variation is captured by a given component. Larger eigenvalues indicate that a component explains more of the shared structure across the included variables.

Because the PCA variables were standardized before estimation, eigenvalues are especially useful for evaluating whether the first principal component captures a meaningful common financial literacy dimension relative to an individual standardized variable.

In [17]:
# Compare eigenvalues across PCA specifications
for name, pca in [("Full sample", pca_full), ("Deduplicated sample", pca_unique)]:

    print(f"\n{name}")
    print("-" * 50)

    print("Eigenvalue of first principal component:")
    print(round(pca.explained_variance_[0], 3))


Full sample
--------------------------------------------------
Eigenvalue of first principal component:
3.863

Deduplicated sample
--------------------------------------------------
Eigenvalue of first principal component:
3.884


The eigenvalue of the first principal component is substantially greater than 1 in both the full and deduplicated samples. Because the variables were standardized before PCA estimation, an eigenvalue above 1 indicates that the component explains more variation than an individual standardized variable.

In this case, the first component explains approximately 3.9 variables worth of shared variance, providing strong evidence that the financial confidence and financial behavior variables contain a meaningful common underlying structure suitable for PCA-based index construction.

The eigenvalues are also nearly identical across the full and deduplicated samples, further supporting the stability of the PCA structure and suggesting that duplicate observations do not materially influence the results.

## Financial Literacy Index Construction Summary

The preceding sections support the following modeling decisions:

- Primary financial literacy measure: `finlit_z_index`
  - Standardizes each included financial confidence and behavior variable before aggregation.
  - Preserves interpretability.
  - Accounts for differences in dispersion and response concentration across items.

- Alternative financial literacy measure: `finlit_pca1`
  - Uses the first principal component from the standardized financial literacy variables.
  - Serves as a robustness check against the preferred Z-score index.
  - PCA diagnostics show stable explained variance, eigenvalues, and loadings across both samples.

- Primary analysis sample: `df_unique`
  - Uses the deduplicated dataset as the main analytic sample.

- Robustness comparison sample: `df_full`
  - Retains the full dataset to check whether duplicate observations materially affect results.

- Outcome variables:
  - `food_insecure`
  - `very_low_security`

## Estimate Baseline Food Insecurity Models

We next estimate preliminary regression models examining the relationship between financial literacy measures and food insecurity outcomes.

The initial specifications focus on the two primary food security outcomes:

- `food_insecure`
- `very_low_security`

The baseline models begin with simple bivariate relationships before introducing additional controls and alternative model specifications in later sections.

### Interpretation and Identification Considerations

The preliminary models in this notebook evaluate empirical associations between financial literacy measures and food insecurity outcomes rather than establish definitive causal effects.

Several sources of endogeneity may affect the estimated relationships. Financial literacy may influence food insecurity outcomes, while food insecurity and financial hardship may also affect financial behavior and self-reported financial confidence. In addition, unobserved characteristics such as long-term socioeconomic conditions, health status, or household support networks may jointly influence both financial literacy and food insecurity.

The baseline models therefore begin with simple bivariate specifications and later introduce additional control variables to reduce potential omitted variable bias arising from observed demographic and socioeconomic characteristics.

Because of these concerns, the estimated relationships should be interpreted primarily as conditional associations rather than definitive causal effects. Later robustness comparisons evaluate whether the main empirical relationships remain stable across alternative specifications, financial literacy measures, and sample definitions.

### Baseline Bivariate Models - Linear Probability Model (LPM)

We begin with simple bivariate linear probability models (LPMs) relating the financial literacy measures to the two food security outcomes.

The initial specifications estimate unconditional relationships between financial literacy and food insecurity before introducing additional demographic and socioeconomic controls in later sections. At this stage, the models are intended to provide transparent baseline comparisons across financial literacy measures, outcome definitions, and analysis samples.

Because the outcomes are binary, the LPM coefficients can be interpreted approximately as percentage-point changes in the probability of the outcome associated with a one-unit increase in the corresponding financial literacy measure.

In [18]:
# Import regression tools
import statsmodels.formula.api as smf

In [19]:
# Estimate baseline LPMs across outcomes, financial literacy measures, and samples
outcomes = ["food_insecure", "very_low_security"]
finlit_measures = ["finlit_z_index", "finlit_pca1"]
samples = {
    "Full sample": df_full,
    "Deduplicated sample": df_unique
}

for sample_name, data in samples.items():
    print(f"\n{sample_name}")
    print("=" * 60)
    
    for outcome in outcomes:
        print(f"\nOutcome: {outcome}")
        print("-" * 60)
        
        for measure in finlit_measures:
            model = smf.ols(
                formula=f"{outcome} ~ {measure}",
                data=data
            ).fit(cov_type="HC1")
            
            coef = model.params[measure]
            pval = model.pvalues[measure]
            
            print(f"{measure}: coef = {coef:.3f}, p-value = {pval:.3f}")


Full sample

Outcome: food_insecure
------------------------------------------------------------
finlit_z_index: coef = -0.113, p-value = 0.000
finlit_pca1: coef = -0.034, p-value = 0.000

Outcome: very_low_security
------------------------------------------------------------
finlit_z_index: coef = -0.083, p-value = 0.000
finlit_pca1: coef = -0.025, p-value = 0.000

Deduplicated sample

Outcome: food_insecure
------------------------------------------------------------
finlit_z_index: coef = -0.158, p-value = 0.000
finlit_pca1: coef = -0.049, p-value = 0.000

Outcome: very_low_security
------------------------------------------------------------
finlit_z_index: coef = -0.101, p-value = 0.000
finlit_pca1: coef = -0.031, p-value = 0.000


The baseline linear probability models produce highly consistent results across financial literacy measures, food security outcomes, and analysis samples.

Across all specifications, higher financial literacy is associated with lower probabilities of both food insecurity and very low food security. All estimated coefficients are negative and statistically significant at conventional levels.

The estimated relationships are also broadly stable across the full and deduplicated samples, further supporting the robustness of the findings to duplicate removal.

The standardized Z-score index generally produces larger coefficient magnitudes than the PCA-based alternative index. This is expected because the two indices are constructed differently and operate on different scales. At this stage, the primary takeaway is the consistency in direction, statistical significance, and overall empirical pattern across alternative financial literacy measures.

### Baseline Bivariate Models - Logit
Because the outcome variables are binary, we next estimate logit models as a nonlinear alternative to the linear probability model. Logit models estimate the relationship between financial literacy and the probability of food insecurity while constraining predicted probabilities to fall between 0 and 1.

At this stage, the logit models are used primarily to compare the direction and consistency of the relationships found in the LPM results.

In [20]:
# Estimate baseline logit models across outcomes, financial literacy measures, and samples
for sample_name, data in samples.items():
    print(f"\n{sample_name}")
    print("=" * 60)
    
    for outcome in outcomes:
        print(f"\nOutcome: {outcome}")
        print("-" * 60)
        
        for measure in finlit_measures:
            model = smf.logit(
                formula=f"{outcome} ~ {measure}",
                data=data
            ).fit(disp=False)
            
            coef = model.params[measure]
            pval = model.pvalues[measure]
            
            print(f"{measure}: coef = {coef:.3f}, p-value = {pval:.3f}")


Full sample

Outcome: food_insecure
------------------------------------------------------------
finlit_z_index: coef = -0.519, p-value = 0.000
finlit_pca1: coef = -0.155, p-value = 0.000

Outcome: very_low_security
------------------------------------------------------------
finlit_z_index: coef = -0.788, p-value = 0.000
finlit_pca1: coef = -0.242, p-value = 0.000

Deduplicated sample

Outcome: food_insecure
------------------------------------------------------------
finlit_z_index: coef = -0.731, p-value = 0.000
finlit_pca1: coef = -0.226, p-value = 0.000

Outcome: very_low_security
------------------------------------------------------------
finlit_z_index: coef = -0.938, p-value = 0.000
finlit_pca1: coef = -0.294, p-value = 0.000


The primary takeaway at this stage is the consistency in direction, statistical significance, and overall empirical pattern across model types, financial literacy measures, outcome definitions, and sample definitions.

### Logit Average Marginal Effects

Because raw logit coefficients are reported in log-odds units, we next estimate average marginal effects. These translate the logit results into approximate changes in predicted probability, making the results easier to compare with the linear probability models.

In [21]:
# Estimate average marginal effects from baseline logit models
for sample_name, data in samples.items():
    print(f"\n{sample_name}")
    print("=" * 60)
    
    for outcome in outcomes:
        print(f"\nOutcome: {outcome}")
        print("-" * 60)
        
        for measure in finlit_measures:
            model = smf.logit(
                formula=f"{outcome} ~ {measure}",
                data=data
            ).fit(disp=False)
            
            margins = model.get_margeff(at="overall")
            margeff_summary = margins.summary_frame()
            
            ame = margeff_summary.loc[measure, "dy/dx"]
            pval = margeff_summary.loc[measure, "Pr(>|z|)"]
            
            print(f"{measure}: AME = {ame:.3f}, p-value = {pval:.3f}")


Full sample

Outcome: food_insecure
------------------------------------------------------------
finlit_z_index: AME = -0.106, p-value = 0.000
finlit_pca1: AME = -0.032, p-value = 0.000

Outcome: very_low_security
------------------------------------------------------------
finlit_z_index: AME = -0.068, p-value = 0.000
finlit_pca1: AME = -0.021, p-value = 0.000

Deduplicated sample

Outcome: food_insecure
------------------------------------------------------------
finlit_z_index: AME = -0.143, p-value = 0.000
finlit_pca1: AME = -0.045, p-value = 0.000

Outcome: very_low_security
------------------------------------------------------------
finlit_z_index: AME = -0.079, p-value = 0.000
finlit_pca1: AME = -0.025, p-value = 0.000


The average marginal effects from the logit models are highly consistent with the earlier linear probability model estimates. Across all specifications, higher financial literacy and financial capability are associated with lower predicted probabilities of both food insecurity and very low food security.

Importantly, the marginal effect estimates are similar in magnitude to the corresponding LPM coefficients, suggesting that the substantive conclusions are not sensitive to the choice between linear and nonlinear probability models.

The estimated effects are again broadly stable across both the full and deduplicated samples, providing additional evidence that the observed relationships are robust to duplicate removal and alternative model specifications.

### Add Basic Control Variables

The baseline models above examine the unconditional relationship between financial literacy and food insecurity. We next introduce a small set of basic control variables to assess whether this relationship remains after accounting for observable demographic, socioeconomic, and health-related differences across respondents.

At this stage, the goal is not to estimate a final causal model. Instead, these adjusted models provide an initial robustness check for whether the financial literacy relationship persists after including obvious potential confounders.

The preliminary control variables include:

1. `qs3`: respondent age
2. `q1`: education level
3. `q4`: household income category
4. `q23`: self-rated health status

These variables are included because they are plausibly related to both financial literacy and food insecurity. Controlling for them helps reduce omitted variable bias from observed respondent characteristics, though it does not fully eliminate concerns about endogeneity or establish causal effects.

In [22]:
# Candidate control variables for preliminary adjusted models
control_vars = ["qs3", "q1", "q4", "q23"]

### Inspect Control Variables

Before estimating adjusted models, we inspect the control variables in both analysis samples. This step checks whether the controls contain missing values, unexpected response codes, or distributional patterns that affect how they should be included in the regression models.

Because `q1`, `q4`, and `q23` are categorical survey responses, they should be reviewed carefully before modeling rather than treated automatically as continuous numeric variables.

In [23]:
# Compact inspection of control variables
for name, data in [("Full sample", df_full), ("Deduplicated sample", df_unique)]:
    print(f"\n{name}")
    print("-" * 50)

    print("\nqs3 (Age)")
    print("Missing:", data["qs3"].isna().sum())
    print(data["qs3"].describe().round(2))

    print("\nCategorical controls")
    display(
        data[["q1", "q4", "q23"]]
        .agg(["count", "nunique", "min", "max"])
        .T
    )


Full sample
--------------------------------------------------

qs3 (Age)
Missing: 31
count    769.00
mean      73.62
std        7.48
min       60.00
25%       68.00
50%       73.00
75%       79.00
max       95.00
Name: qs3, dtype: float64

Categorical controls


,count,nunique,min,max
q1,800,6,1,6
q4,800,16,1,16
q23,800,6,1,6



Deduplicated sample
--------------------------------------------------

qs3 (Age)
Missing: 24
count    531.00
mean      73.55
std        7.39
min       60.00
25%       68.00
50%       73.00
75%       79.00
max       95.00
Name: qs3, dtype: float64

Categorical controls


,count,nunique,min,max
q1,555,6,1,6
q4,555,16,1,16
q23,555,6,1,6


The preliminary control variables appear broadly suitable for regression analysis across both samples.

Age (`qs3`) exhibits a reasonable distribution for a senior population sample, with highly similar summary statistics across the full and deduplicated datasets. The categorical controls (`q1`, `q4`, and `q23`) also appear internally consistent and contain expected response ranges based on the survey coding structure.

Some missingness is present for age, meaning the adjusted regression models will use slightly smaller estimation samples than the baseline bivariate models. At this stage, the control variables are included primarily to assess whether the observed financial literacy relationships remain stable after accounting for basic demographic, socioeconomic, and health-related differences across respondents.

### Prepare Control Variables for Modeling

The adjusted regression models next incorporate the preliminary demographic, socioeconomic, and health-related controls identified above.

Age (`qs3`) is included as a continuous variable, while education (`q1`), household income (`q4`), and self-rated health (`q23`) are treated as categorical controls in the regression specifications. Treating these variables categorically avoids imposing unrealistic linearity assumptions across survey response categories.

In [24]:
# Construct model-ready control specification; qs3 enters continuously, while q1, q4, and q23 are treated as categorical variables using C()

controls = "qs3 + C(q1) + C(q4) + C(q23)"

### Estimate Controlled Linear Probability Models

We next estimate adjusted linear probability models that include the basic control variables. These models assess whether the relationship between financial literacy and food insecurity remains after accounting for observed differences in age, education, household income, and self-rated health.

In [25]:
# Estimate controlled LPM models across outcomes, financial literacy indices, and samples

controlled_lpm_results = []

for sample_name, data in [("Full sample", df_full), ("Deduplicated sample", df_unique)]:
    for outcome in outcomes:
        for measure in finlit_measures:

            formula = f"{outcome} ~ {measure} + {controls}"

            model = smf.ols(formula, data=data).fit(cov_type="HC1")

            controlled_lpm_results.append({
                "sample": sample_name,
                "outcome": outcome,
                "finlit_measure": measure,
                "coef": model.params[measure],
                "se": model.bse[measure],
                "p_value": model.pvalues[measure],
                "n_obs": int(model.nobs)
            })

controlled_lpm_results_df = pd.DataFrame(controlled_lpm_results)

controlled_lpm_results_df

,sample,outcome,finlit_measure,coef,se,p_value,n_obs
0,Full sample,food_insecure,finlit_z_index,-0.038886,0.024187,0.107900,769
1,Full sample,food_insecure,finlit_pca1,-0.010121,0.007931,0.201900,769
2,Full sample,very_low_security,finlit_z_index,-0.049208,0.019714,0.012554,769
3,Full sample,very_low_security,finlit_pca1,-0.014617,0.006384,0.022055,769
4,Deduplicated sample,food_insecure,finlit_z_index,-0.078764,0.028802,0.006243,531
5,Deduplicated sample,food_insecure,finlit_pca1,-0.023613,0.009455,0.012508,531
6,Deduplicated sample,very_low_security,finlit_z_index,-0.067301,0.023876,0.004820,531
7,Deduplicated sample,very_low_security,finlit_pca1,-0.020781,0.007745,0.007294,531


The adjusted linear probability models continue to show generally negative relationships between financial literacy and food insecurity outcomes after introducing the preliminary demographic, socioeconomic, and health-related controls.

The estimated relationships are weaker than in the earlier bivariate specifications, suggesting that part of the baseline association between financial literacy and food insecurity is explained by observable respondent characteristics such as age, education, income, and self-rated health. However, several financial literacy coefficients remain statistically significant after adjustment, particularly for the `very_low_security` outcome and within the deduplicated sample.

Across both samples, the standardized Z-score index and the PCA-based alternative index continue to produce consistent directional results. The coefficient magnitudes remain relatively stable across the full and deduplicated datasets, further supporting the robustness of the observed relationships to duplicate removal.

Because several categorical controls include sparse response categories, HC1 robust standard errors are used in the preliminary adjusted LPMs. This avoids instability from high-leverage observations while still allowing heteroskedasticity-robust inference.

At this stage, the adjusted models should still be interpreted as conditional associations rather than definitive causal effects. The purpose of the control variables is to reduce omitted variable bias from observed characteristics, not to fully resolve remaining endogeneity concerns.

### Controlled Logit Models

Controlled logit models were explored as a nonlinear extension of the adjusted linear probability specifications. However, several controlled logit specifications produced singular matrix and separation-related estimation issues after including the full set of categorical controls.

This instability likely arises from sparse response categories within some of the control variables, particularly when combined with relatively smaller subgroup counts in the deduplicated sample. Under these conditions, certain combinations of predictors can nearly or perfectly predict the outcome, making stable maximum likelihood estimation difficult.

Because the primary purpose of the adjusted models at this stage is to evaluate whether the financial literacy relationships remain after accounting for observable respondent characteristics, the controlled linear probability models are retained as the main adjusted specifications.

Later analysis may revisit controlled nonlinear probability models after collapsing sparse control categories or simplifying the control structure.

### Control Variable Refinement

The controlled models suggest that the current control structure may need refinement in later analysis. In particular, the categorical controls include several sparse response categories, which contributed to instability in the controlled logit specifications.

A later version of the analysis may collapse sparse categories for variables such as education, income, and self-rated health before estimating final adjusted models. This would reduce separation risk, improve model stability, and make the controlled specifications easier to interpret.

For the preliminary analysis, the controlled LPMs are retained because they provide a stable first adjustment for observed demographic, socioeconomic, and health-related differences.

### Preliminary Model Takeaways

The preliminary models show a consistent negative association between financial literacy and food insecurity. Across alternative samples, outcome definitions, index construction methods, and model specifications, higher financial literacy is generally associated with lower food insecurity risk.

The relationship weakens after adding basic controls, suggesting that part of the baseline association reflects observable differences in age, education, income, and health. However, several adjusted estimates remain negative and statistically significant, especially in the deduplicated sample.

These results should be interpreted as conditional associations, not causal effects. Overall, the preliminary evidence supports continuing to refine the model specification while treating financial literacy as a meaningful predictor of food insecurity outcomes.

## Summary of Preliminary Model Results

The table below summarizes the main preliminary model estimates across samples, outcomes, financial literacy indices, and model specifications.

In [26]:
# Create final preliminary model summary table from scratch

summary_results = []

# 1. Baseline LPM
for sample_name, data in [("Full sample", df_full), ("Deduplicated sample", df_unique)]:
    for outcome in outcomes:
        for measure in finlit_measures:
            model = smf.ols(
                formula=f"{outcome} ~ {measure}",
                data=data
            ).fit(cov_type="HC1")

            summary_results.append({
                "specification": "Baseline LPM",
                "sample": sample_name,
                "outcome": outcome,
                "finlit_measure": measure,
                "estimate": model.params[measure],
                "se": model.bse[measure],
                "p_value": model.pvalues[measure],
                "n_obs": int(model.nobs)
            })

# 2. Baseline Logit AME
for sample_name, data in [("Full sample", df_full), ("Deduplicated sample", df_unique)]:
    for outcome in outcomes:
        for measure in finlit_measures:
            model = smf.logit(
                formula=f"{outcome} ~ {measure}",
                data=data
            ).fit(disp=False)

            ame = model.get_margeff(at="overall").summary_frame()

            summary_results.append({
                "specification": "Baseline Logit AME",
                "sample": sample_name,
                "outcome": outcome,
                "finlit_measure": measure,
                "estimate": ame.loc[measure, "dy/dx"],
                "se": ame.loc[measure, "Std. Err."],
                "p_value": ame.loc[measure, "Pr(>|z|)"],
                "n_obs": int(model.nobs)
            })

# 3. Controlled LPM
for sample_name, data in [("Full sample", df_full), ("Deduplicated sample", df_unique)]:
    for outcome in outcomes:
        for measure in finlit_measures:
            model = smf.ols(
                formula=f"{outcome} ~ {measure} + {controls}",
                data=data
            ).fit(cov_type="HC1")

            summary_results.append({
                "specification": "Controlled LPM",
                "sample": sample_name,
                "outcome": outcome,
                "finlit_measure": measure,
                "estimate": model.params[measure],
                "se": model.bse[measure],
                "p_value": model.pvalues[measure],
                "n_obs": int(model.nobs)
            })

model_summary_table = pd.DataFrame(summary_results)

model_summary_table = model_summary_table.round({
    "estimate": 3,
    "se": 3,
    "p_value": 3
})

model_summary_table

,specification,sample,outcome,finlit_measure,estimate,se,p_value,n_obs
0,Baseline LPM,Full sample,food_insecure,finlit_z_index,-0.113,0.026,0.000,800
1,Baseline LPM,Full sample,food_insecure,finlit_pca1,-0.034,0.009,0.000,800
2,Baseline LPM,Full sample,very_low_security,finlit_z_index,-0.083,0.020,0.000,800
3,Baseline LPM,Full sample,very_low_security,finlit_pca1,-0.025,0.006,0.000,800
4,Baseline LPM,Deduplicated sample,food_insecure,finlit_z_index,-0.158,0.030,0.000,555
5,Baseline LPM,Deduplicated sample,food_insecure,finlit_pca1,-0.049,0.010,0.000,555
6,Baseline LPM,Deduplicated sample,very_low_security,finlit_z_index,-0.101,0.024,0.000,555
7,Baseline LPM,Deduplicated sample,very_low_security,finlit_pca1,-0.031,0.008,0.000,555
8,Baseline Logit AME,Full sample,food_insecure,finlit_z_index,-0.106,0.023,0.000,800
9,Baseline Logit AME,Full sample,food_insecure,finlit_pca1,-0.032,0.008,0.000,800


The summary table consolidates the main preliminary model results across baseline LPMs, baseline logit average marginal effects, and controlled LPMs.

Across the baseline specifications, higher financial literacy is consistently associated with lower probabilities of both `food_insecure` and `very_low_security`. The baseline LPM estimates and logit average marginal effects are similar in sign and magnitude, suggesting that the main results are not highly sensitive to the choice between linear and nonlinear probability models.

After adding controls, the estimated relationships become smaller in magnitude, indicating that observable characteristics such as age, education, income, and health explain part of the unconditional association. However, several controlled estimates remain negative and statistically significant, especially in the deduplicated sample.

Overall, the table supports the main preliminary finding that higher financial literacy is associated with lower food insecurity risk, while also showing that the relationship should be interpreted cautiously as a conditional association rather than a causal effect.

# Robustness Summary

Several robustness checks were performed throughout the preliminary analysis to evaluate whether the observed relationship between financial literacy and food insecurity remains stable across alternative samples, index construction methods, and model specifications.

The analysis compared results across:

1. Full and deduplicated analysis samples.
2. Standardized Z-score and PCA-based financial literacy indices.
3. Linear probability and nonlinear logit model specifications.
4. Food insecurity and very low food security outcome definitions.
5. Baseline and adjusted regression specifications.

Across nearly all specifications, higher financial literacy remained associated with lower probabilities of food insecurity and very low food security. The estimated relationships were generally stable in direction and statistical significance across alternative modeling approaches and sample definitions.

The adjusted models showed smaller coefficient magnitudes than the baseline bivariate specifications, suggesting that part of the observed relationship is explained by demographic, socioeconomic, and health-related differences across respondents. However, several adjusted relationships remained statistically significant after introducing controls.

Some instability emerged in the controlled nonlinear probability models due to sparse categorical control categories and separation-related estimation issues. Because of this, the adjusted linear probability models were retained as the primary controlled specifications at this stage.

Overall, the consistency of the results across multiple robustness dimensions suggests that the observed empirical relationships are not driven solely by duplicate observations, a single index construction approach, or a particular probability model specification.

# Variable Info — Final Modeling Section (Ruben)

The completed notebook now establishes the model-ready variables and naming conventions. It is important that the final modeling section uses the same variable names exactly as they appear here so our notebooks stay consistent and we avoid errors from duplicate or renamed variables.

- Primary financial literacy measure: `finlit_z_index`

  - Use this as the main financial literacy index in final models.

- Alternative financial literacy measure: `finlit_pca1`

  - Use this as a robustness check.

- Main outcomes:

  - `food_insecure`

  - `very_low_security`

- Primary analysis sample: `df_unique`

  - Use the deduplicated dataset for main models.

- Robustness comparison sample: `df_full`

  - Use the full dataset to check whether duplicate observations affect results.

- Control variables:

  - `qs3`: age

  - `q1`: education

  - `q4`: household income

  - `q23`: self-rated health

- Current control specification:

  - `qs3 + C(q1) + C(q4) + C(q23)`

- Modeling note:

  - Controlled LPMs are currently stable.

  - Controlled logit models had singular matrix issues, likely because some categorical control groups are sparse.

  - Next step is to collapse sparse control categories before estimating final adjusted models.

- Interpretation:

  - Treat results as conditional associations, not definitive causal effects.

# Notes on Instrumental Variables

We do not use an instrumental variables approach at this stage because the dataset does not appear to contain a clearly defensible instrument.

A valid instrument would need to:

1. Strongly predict financial literacy.
2. Affect food insecurity only through financial literacy.

The second condition is the main challenge. `q43`, which measures parental or guardian financial capability during childhood, is the closest possible candidate. However, it likely also captures broader family background and long-run socioeconomic conditions that may directly affect later food insecurity. This makes the exclusion restriction difficult to defend.

Because of this, a weakly justified IV strategy could reduce the credibility of the analysis. Instead, the current approach emphasizes transparent variable construction, alternative financial literacy indices, robustness comparisons across samples and models, and careful interpretation of results as conditional associations rather than causal effects.

# Summary of Notebook Revisions

The notebook was substantially revised to strengthen the workflow, clarify variable construction, address professor feedback, and improve the robustness of the preliminary modeling section.

Key updates include:

- Added a clearer workflow order to the notebook introduction.
- Added a crosstab under `Construct Food Insecurity Outcome Variables` to verify that `very_low_security` is properly nested within `food_insecure`.
- Clarified the distinction between financial literacy, financial capability, financial confidence, and financial behavior in the financial literacy variable construction section.
- Improved the discussion of `q43`, including why it differs conceptually from the current financial literacy measures and why its response structure requires caution.
- Added a financial literacy missingness subsection and confirmed that no financial literacy responses are missing.
- Tightened the `Construct Preliminary Financial Literacy Measures` section and clarified that these measures are descriptive benchmarks before standardized index construction.
- Strengthened the `Examine Relationships Among Financial Literacy Variables` section to better explain why the correlation structure supports index construction.
- Added a full `Construct Z-Score Financial Literacy Index` section with explanation, code, output, and interpretation.
- Rewrote the opening to the PCA section to clarify that PCA is used as an alternative financial literacy index.
- Expanded the PCA standardization explanation.
- Added an eigenvalue subsection, as requested, with interpretation of what eigenvalues measure and why they matter.
- Added a `Financial Literacy Index Construction Summary` identifying the main index, alternative index, samples, and outcome variables.
- Added an `Interpretation and Identification Considerations` subsection to address endogeneity and clarify that results should be interpreted as conditional associations.
- Revised the LPM and logit baseline sections to use the correct financial literacy variables: `finlit_z_index` and `finlit_pca1`.
- Improved the explanations before and after the LPM, logit, and marginal effects models.
- Added and inspected basic control variables.
- Estimated controlled LPM models and explained why controlled logit models were not retained due to sparse categories and estimation instability.
- Added a final model summary table consolidating baseline LPM, baseline logit AME, and controlled LPM results.

Furthermore, the section from `Baseline Bivariate Models` onward was overhauled to create a cleaner and more defensible empirical workflow.